# Analisis Data Gempa Bumi

Notebook ini memproses data gempa bumi dari file `database.csv`. Data yang dipakai hanya kejadian bertipe **Earthquake**, lalu diberi kolom waktu, dibagi per periode, dan divisualisasikan.

Target yang diprediksi adalah **Magnitude**. Fitur yang dipakai untuk prediksi adalah Latitude, Longitude, Depth, Depth Seismic Stations, Year, Month, dan Hour.

## Import Library

Mengimpor library yang dibutuhkan:
- **numpy** dan **pandas** untuk mengolah data tabel.
- **matplotlib** untuk membuat grafik.
- **scikit-learn** untuk model dasar (`DummyRegressor`, `Ridge`), imputasi nilai kosong, scaling, pipeline, dan metrik evaluasi (MAE, MSE, R²).

In [2]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


## Membaca Data dan Filter Kejadian

Membaca file `database.csv` ke dalam DataFrame lalu menyimpan jumlah baris awal. Setelah itu data difilter hanya untuk baris dengan `Type == Earthquake`, sehingga ledakan nuklir atau jenis kejadian lain tidak ikut. `.copy()` dipakai agar hasil filter berdiri sendiri dan tidak memunculkan peringatan pandas saat diubah nanti. Kolom-kolom yang tersedia juga ditampilkan untuk memastikan nama kolomnya benar.

In [3]:

Target = "Magnitude"
Fitur = [
    "Latitude",
    "Longitude",
    "Depth",
    "Depth Seismic Stations",
    "Year",
    "Month",
    "Hour"
]

data = pd.read_csv("database.csv")
print(f"Baris awal            : {len(data)}")

data = data[data["Type"] == "Earthquake"].copy()
print(f"Setelah filter Earthquake: {len(data)}")
print(data.columns.tolist())



Baris awal            : 23412
Setelah filter Earthquake: 23232
['Date', 'Time', 'Latitude', 'Longitude', 'Type', 'Depth', 'Depth Error', 'Depth Seismic Stations', 'Magnitude', 'Magnitude Type', 'Magnitude Error', 'Magnitude Seismic Stations', 'Azimuthal Gap', 'Horizontal Distance', 'Horizontal Error', 'Root Mean Square', 'ID', 'Source', 'Location Source', 'Magnitude Source', 'Status']


## Menggabungkan Date dan Time menjadi Datetime

Kolom `Date` dan `Time` digabung menjadi satu string lalu diubah menjadi kolom `datetime` dengan format `%m/%d/%Y %H:%M:%S`. Parameter `errors=coerce` membuat baris yang gagal dibaca menjadi `NaT` (nilai kosong) alih-alih menghentikan program. Baris-baris tersebut kemudian dibuang dengan `dropna`, dan `info()` menampilkan ringkasan tipe data serta jumlah nilai non-null.

Baris yang dicomment (`# print(...)`) adalah versi untuk menghitung berapa baris yang gagal diparse, bisa diaktifkan jika ingin melihat jumlahnya.

In [4]:

data["datetime"] = pd.to_datetime(
    data["Date"] + " " + data["Time"],
    format="%m/%d/%Y %H:%M:%S",
    errors="coerce",
)
# print(f"Tanggal/waktu tidak valid: {data['datetime'].isna().sum()}")
data = data.dropna(subset=["datetime"]).copy()
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 23229 entries, 0 to 23411
Data columns (total 22 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   Date                        23229 non-null  object        
 1   Time                        23229 non-null  object        
 2   Latitude                    23229 non-null  float64       
 3   Longitude                   23229 non-null  float64       
 4   Type                        23229 non-null  object        
 5   Depth                       23229 non-null  float64       
 6   Depth Error                 4448 non-null   float64       
 7   Depth Seismic Stations      7083 non-null   float64       
 8   Magnitude                   23229 non-null  float64       
 9   Magnitude Type              23226 non-null  object        
 10  Magnitude Error             315 non-null    float64       
 11  Magnitude Seismic Stations  2460 non-null   float64       


## Memecah Datetime menjadi Year, Month, dan Hour

Mengambil bagian tahun, bulan, dan jam dari kolom `datetime` dan menyimpannya sebagai kolom terpisah. Kolom ini dibutuhkan sebagai fitur model (Year, Month, Hour) dan untuk pengelompokan per tahun pada grafik berikutnya.

In [5]:
data["Year"] = data["datetime"].dt.year
data["Month"] = data["datetime"].dt.month
data["Hour"] = data["datetime"].dt.hour

## Jumlah Kejadian per Tahun dan Kelengkapan Data

Menghitung jumlah gempa per tahun dengan `groupby(Year).size()`. Variabel `kosong_pct` dihitung untuk persentase nilai kosong pada kolom kualitas per tahun, sehingga bisa dilihat apakah pencatatan data membaik seiring waktu.

Grafik yang dihasilkan berupa diagram batang untuk jumlah kejadian per tahun dan disimpan sebagai `gambar_z_kejadian_per_tahun.png`.

Catatan: di kode ini variabel `kolom_kualitas` belum didefinisikan, jadi nama kolom kualitas harus ditentukan dulu sebelum sel ini dijalankan. Persentase `kosong_pct` juga belum ditampilkan di grafik, baru dihitung.

In [6]:

per_tahun = data.groupby("Year").size()
kosong_pct = data.groupby("Year")[kolom_kualitas].apply(
    lambda s: s.isna().mean() * 100
)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(per_tahun.index, per_tahun.values, color="steelblue",
        label="Jumlah kejadian")
ax1.set_xlabel("Tahun")
ax1.set_ylabel("Jumlah Kejadian")

lines1, labels1 = ax1.get_legend_handles_labels()
ax1.legend(lines1, labels1 , loc="upper left")
ax1.set_title("Jumlah Kejadian per Tahun dan Kelengkapan Data")
plt.tight_layout()
plt.savefig("gambar_z_kejadian_per_tahun.png", dpi=300)

NameError: name 'kolom_kualitas' is not defined

## Menyimpan Data Bersih

Memilih hanya kolom `datetime`, fitur, dan target dari data yang sudah dibersihkan, lalu menyimpannya ke `gempa_clean.csv` tanpa kolom index. File ini bisa dipakai lagi di notebook lain tanpa mengulang proses pembersihan.

In [ ]:
data = data[["datetime"] + Fitur + [Target]].copy()

data.to_csv("gempa_clean.csv", index=False)

## Pembagian Data Train, Validation, dan Test

Data dibagi berdasarkan tahun kejadian, bukan secara acak, agar model diuji pada data yang lebih baru seperti dalam kondisi nyata:
- **Train**: tahun 2000 ke bawah.
- **Validation**: tahun 2001 sampai 2010.
- **Test**: tahun setelah 2010.

Setiap bagian lalu dipisah menjadi `X` (fitur) dan `y` (target). Jumlah baris tiap bagian dicetak untuk memastikan pembagiannya masuk akal.

In [ ]:
year = data["datetime"].dt.year
train = data[year <= 2000]
val = data[(year > 2000) & (year <= 2010)]
test = data[year > 2010]

print(f"\nTrain: {len(train)} | Val: {len(val)} | Test: {len(test)}")

X_train, y_train = train[Fitur], train[Target]
X_val, y_val = val[Fitur], val[Target]
X_test, y_test = test[Fitur], test[Target]

## Distribusi Magnitudo

Membuat histogram magnitudo dengan 30 bin. Garis merah putus-putus menunjukkan rata-rata dan garis oranye menunjukkan median, sehingga terlihat apakah distribusinya condong ke satu sisi. Grafik disimpan sebagai `gambar_x_histogram_magnitudo.png`.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(data["Magnitude"], bins=30, color="steelblue", edgecolor="white")
ax.set_xlabel("Magnitudo")
ax.set_ylabel("Jumlah Kejadian")
ax.axvline(data["Magnitude"].mean(), color="red", linestyle="--",
           label=f"Rata-rata = {data['Magnitude'].mean():.2f}")
ax.axvline(data["Magnitude"].median(), color="orange", linestyle="--",
           label=f"Median = {data['Magnitude'].median():.2f}")
ax.legend()

plt.tight_layout()
plt.savefig("gambar_x_histogram_magnitudo.png", dpi=300)
plt.show()

## Distribusi Kedalaman

Histogram kedalaman gempa dengan 50 bin, ditandai dengan garis median. Grafik ini menunjukkan apakah gempa umumnya dangkal atau dalam. Hasilnya disimpan sebagai `gambar_y_distribusi_kedalaman.png`.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(data["Depth"], bins=50, color="seagreen", edgecolor="white")
ax.axvline(data["Depth"].median(), color="red", linestyle="--",
           label=f"Median = {data['Depth'].median():.1f} km")
ax.set_title("Distribusi Kedalaman Gempa Bumi (1965-2016)")
ax.set_xlabel("Kedalaman (km)")
ax.set_ylabel("Jumlah Kejadian")
ax.legend()
plt.tight_layout()
plt.savefig("gambar_y_distribusi_kedalaman.png", dpi=300)

## Sebaran Spasial Episentrum

Scatter plot dengan Longitude sebagai sumbu X dan Latitude sebagai sumbu Y. Warna setiap titik menunjukkan magnitudo, dengan skala warna kuning ke merah. Grafik ini memperlihatkan di mana gempa paling sering terjadi, biasanya mengikuti batas lempeng tektonik. Hasilnya disimpan sebagai `gambar_w_sebaran_spasial.png`.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sc = ax.scatter(data["Longitude"], data["Latitude"], c=data["Magnitude"],
                cmap="YlOrRd", s=5, alpha=0.6)
plt.colorbar(sc, ax=ax, label="Magnitudo")
ax.set_title("Sebaran Episentrum Gempa Bumi (1965-2016)")
ax.set_xlabel("Bujur")
ax.set_ylabel("Lintang")
plt.tight_layout()
plt.savefig("gambar_w_sebaran_spasial.png", dpi=300)

## Korelasi Kedalaman dan Magnitudo

Menghitung korelasi **Spearman** antara Depth dan Magnitude. Spearman dipilih karena tidak mengasumsikan hubungan yang lurus dan tidak terlalu terpengaruh oleh pencilan. Nilainya mendekati 0 jika hubungannya lemah, dan mendekati 1 atau -1 jika hubungannya kuat.

Hasil korelasi dicetak, lalu dibuat scatter plot Depth vs Magnitude dengan nilai korelasi di judul. Grafik disimpan sebagai `gambar_v_depth_vs_magnitudo.png`.

In [ ]:
korelasi = data["Depth"].corr(data["Magnitude"], method="spearman")
print(f"Korelasi Spearman Depth-Magnitude: {korelasi:.3f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(data["Depth"], data["Magnitude"], s=5, alpha=0.3, color="purple")
ax.set_title(f"Kedalaman vs Magnitudo (Spearman r = {korelasi:.2f})")
ax.set_xlabel("Kedalaman (km)")
ax.set_ylabel("Magnitudo")
plt.tight_layout()
plt.savefig("gambar_v_depth_vs_magnitudo.png", dpi=300)

plt.show()